In [ ]:
from collections import Counter

import numpy as np
import plotly.graph_objects as go

from notebooks.metrics import precision_k

# replace paths
result = precision_k("../../reference_test.csv", "../../predict_test.csv")

scores = [info["score"] for info in result["detail"].values()]
n_files = len(scores)

# Распределение Precision@k

In [ ]:
fig = go.Figure()

fig.add_trace(go.Histogram(x=scores, nbinsx=50, marker_color="#1f77b4", opacity=0.7))

mean_score = np.mean(scores)
median_score = np.median(scores)

fig.add_vline(x=mean_score, line_dash="dash", line_color="red", line_width=2)
fig.add_vline(x=median_score, line_dash="dash", line_color="green", line_width=2)

bad_files = sum(1 for s in scores if s < 0.5)
perfect_files = sum(1 for s in scores if s == 1.0)

fig.add_annotation(
    x=0.98,
    y=0.98,
    xref="paper",
    yref="paper",
    text=(
        f"<b>Статистика (N={n_files:,})</b><br>"
        f"━━━━━━━━━━━━━━━━━━<br>"
        f"📊 Среднее: {mean_score:.4f}<br>"
        f"📊 Медиана: {median_score:.4f}<br>"
        f"❌ Score &lt; 0.5: {bad_files:,} ({100*bad_files/n_files:.1f}%)<br>"
        f"✅ Score = 1.0: {perfect_files:,} ({100*perfect_files/n_files:.1f}%)"
    ),
    showarrow=False,
    font=dict(size=11),
    bgcolor="rgba(255,255,255,0.95)",
    bordercolor="gray",
    borderwidth=1,
    borderpad=6,
    align="left",
    xanchor="right",
    yanchor="top",
)

fig.update_layout(
    title="Распределение Precision@k",
    xaxis_title="Precision@k",
    yaxis_title="Количество файлов",
    xaxis=dict(range=[0, 1], dtick=0.1),
    yaxis_title_font_size=12,
    height=500,
    showlegend=False,
    bargap=0.05,
)

fig.show()

# ТОП худших по Ppecision@k файлов

In [ ]:
N_WORST = 20
worst_indices = np.argsort(scores)[:N_WORST]
worst_files = [list(result["detail"].keys())[i] for i in worst_indices]
worst_scores = [scores[i] for i in worst_indices]

fig = go.Figure(
    go.Bar(
        x=worst_scores,
        y=[f.split("/")[-1][:30] for f in worst_files],
        orientation="h",
        marker_color="red",
        text=[f"{s:.3f}" for s in worst_scores],
        textposition="outside",
    )
)

fig.update_layout(
    title=f"Топ-{N_WORST} худших файлов (score ≤ {worst_scores[-1]:.3f})",
    xaxis_title="Precision@k",
    yaxis_title="Файл",
    xaxis=dict(range=[0, 1]),
    height=600,
)
fig.show()

# Топ лишних ID (extra) — модель предсказывает их слишком часто

In [ ]:
N_EXTRAS = 20

all_extra = []
for info in result["detail"].values():
    all_extra.extend(info["extra"])

extra_counter = Counter(all_extra)
top_extra = sorted(extra_counter.most_common(N_EXTRAS), key=lambda pair: pair[1])

fig = go.Figure(
    go.Bar(
        x=[count for _, count in top_extra],
        y=[str(idx) for idx, _ in top_extra],
        orientation="h",
        marker_color="coral",
        text=[count for _, count in top_extra],
        textposition="outside",
    )
)

fig.update_layout(
    title=f"Топ-{N_EXTRAS} лишних ID (extra) — модель предсказывает их слишком часто",
    xaxis_title="Количество появлений в extra",
    yaxis_title="ID",
    height=600,
)
fig.show()

# Топ пропущенных ID (missing) — модель часто их не находит

In [ ]:
N_MISSING = 20

all_missing = []
for info in result["detail"].values():
    all_missing.extend(info["missing"])

missing_counter = Counter(all_missing)
top_missing = sorted(missing_counter.most_common(N_MISSING), key=lambda pair: pair[1])

fig = go.Figure(
    go.Bar(
        x=[count for _, count in top_missing],
        y=[idx for idx, _ in top_missing],
        orientation="h",
        marker_color="lightblue",
        text=[count for _, count in top_missing],
        textposition="outside",
    )
)

fig.update_layout(
    title=f"Топ-{N_MISSING} пропущенных ID (missing) — модель часто их не находит",
    xaxis_title="Количество пропусков",
    yaxis_title="ID",
    height=600,
)
fig.show()